In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import cvxpy as cp
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def load_data(file_path):
    """
    载入数据（基础和市值行业等数据，只载入相关数据提高效率）[cite: 1]
    """
    # TODO: 根据实际情况读取 CSV/数据库/API 数据
    # df = pd.read_csv(file_path, usecols=['date', 'ticker', 'close', 'open', 'is_st', 'suspended', 'limit_up', 'limit_down', 'list_days', 'market_cap', 'industry'])
    df = pd.DataFrame() 
    return df

In [5]:
daily_data = pd.read_parquet('daily_data.parquet')
industry_data = pd.read_parquet('industry.parquet')
st_data = pd.read_parquet('st.parquet')
tp_data = pd.read_parquet('停牌.parquet')

In [3]:
daily_data

AVERAGE_PRICE  DY-ADJ_AF-CLOSE_PRICE_2  \
DATE       CODE                                             
2010-01-04 000001     989.634107                  978.291   
           000002    1080.801182                 1074.195   
           000004       0.000000                   68.339   
           000005      58.661610                   58.858   
           000006     224.459023                  222.885   
...                          ...                      ...   
2021-12-31 688799      41.079319                   41.170   
           688800     141.484335                  139.280   
           688819      43.343751                   43.430   
           688981      53.046054                   52.990   
           689009      69.330011                   70.070   

                   DY-ADJ_AF-HIGHEST_PRICE_2  DY-ADJ_AF-LOWEST_PRICE_2  \
DATE       CODE                                                          
2010-01-04 000001                   1014.188                   977.053   
           000002                   1101.557                  1074.195   
           000004                      0.000                     0.000   
           000005                     59.448                    58.072   
           000006                    227.495                   222.685   
...                                      ...                       ...   
2021-12-31 688799                     41.720                    40.020   
           688800                    149.000                   137.500   
           688819                     43.512                    43.014   
           688981                     53.450                    52.740   
           689009                     71.250                    67.300   

                   DY-ADJ_AF-OPEN_PRICE_2  DY-ADJ_AF-TURNOVER_VOL  \
DATE       CODE                                                     
2010-01-04 000001                1011.712                  806410   
           000002                1099.530                 1749491   
           000004                   0.000                       0   
           000005                  59.055                 3444239   
           000006                 227.094                  482074   
...                                   ...                     ...   
2021-12-31 688799                  40.020                 1587397   
           688800                 145.370                  740989   
           688819                  43.126                 1525722   
           688981                  52.900                14307271   
           689009                  69.000                 2082295   

                   DY-BASIC-DEAL_AMOUNT  DY-BASIC-MARKET_VALUE  \
DATE       CODE                                                  
2010-01-04 000001               20836.0           7.362983e+10   
           000002               68592.0           1.165492e+11   
           000004                   0.0           8.397668e+08   
           000005               12059.0           5.476858e+09   
           000006                4417.0           5.639878e+09   
...                                 ...                    ...   
2021-12-31 688799                2941.0           3.861746e+09   
           688800                2360.0           1.504224e+10   
           688819                2344.0           4.160588e+10   
           688981               17526.0           4.188254e+11   
           689009                4625.0           4.960164e+10   

                   DY-BASIC-NEG_MARKET_VALUE  DY-BASIC-TURNOVER_RATE  \
DATE       CODE                                                        
2010-01-04 000001               6.933075e+10                  0.0083   
           000002               1.023546e+11                  0.0100   
           000004               7.030741e+08                  0.0000   
           000005               5.473321e+09                  0.0245   
           000006               5.485748e+09                  0.0128   
.

In [6]:
industry_data

TYPE_ID LEVEL1_NAME LEVEL2_NAME LEVEL3_NAME
CODE   DATE                                                        
000001 20100104  DY010321330101        金融服务          银行          银行
       20100105  DY010321330101        金融服务          银行          银行
       20100106  DY010321330101        金融服务          银行          银行
       20100107  DY010321330101        金融服务          银行          银行
       20100108  DY010321330101        金融服务          银行          银行
...                         ...         ...         ...         ...
689009 20211227  DY010321110401          汽车      摩托车及其他      其他运输设备
       20211228  DY010321110401          汽车      摩托车及其他      其他运输设备
       20211229  DY010321110401          汽车      摩托车及其他      其他运输设备
       20211230  DY010321110401          汽车      摩托车及其他      其他运输设备
       20211231  DY010321110401          汽车      摩托车及其他      其他运输设备

[9012858 rows x 4 columns]

In [7]:
st_data

,000001,000002,000003,000004,000005,000006,000007,000008,000009,000010,...,688787,688788,688789,688793,688798,688799,688800,688819,688981,689009
20100104,None,None,T,Y,NaN,None,S,S,None,S,...,None,None,None,None,None,None,None,None,None,None
20100105,None,None,T,Y,NaN,None,S,S,None,S,...,None,None,None,None,None,None,None,None,None,None
20100106,None,None,T,Y,NaN,None,S,S,None,S,...,None,None,None,None,None,None,None,None,None,None
20100107,None,None,T,Y,NaN,None,S,S,None,S,...,None,None,None,None,None,None,None,None,None,None
20100108,None,None,T,Y,NaN,None,S,S,None,S,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20221226,None,None,T,S,S,None,NaN,NaN,None,NaN,...,None,None,None,None,None,None,None,None,None,None
20221227,None,None,T,S,S,None,NaN,NaN,None,NaN,...,None,None,None,None,None,None,None,None,None,None
20221228,None,None,T,S,S,None,NaN,NaN,None,NaN,...,None,None,None,None,None,None,None,None,None,None
20221229,None,None,T,S,S,None,NaN,NaN,None,NaN,...,None,None,None,None,None,None,None,None,None,None


In [10]:
tp_data

,股票代码,日期,是否停牌
0,000001,2010-01-04,0
1,000002,2010-01-04,0
2,000004,2010-01-04,1
3,000005,2010-01-04,0
4,000006,2010-01-04,0
...,...,...,...
8730451,871396,2021-12-31,0
8730452,871553,2021-12-31,0
8730453,871642,2021-12-31,0
8730454,871981,2021-12-31,0
